# **Análisis Predictivo de Ventas por Tipo de Producto**

# 1. OBTENCIÓN DE DATOS DESDE KAGGLE

En esta sección se configura el acceso a la API de Kaggle desde Google Colab mediante el archivo de credenciales kaggle.json`.

Este paso permite acceder a datasets públicos de manera reproducible y automatizada.

**NOTA:**
Este procedimiento debe ejecutarse una única vez por sesión.
Tras la carga del archivo, se recomienda reiniciar el entorno de ejecución para aplicar correctamente la configuración.


In [1]:
from google.colab import files
import os
import shutil

print("Sube tu archivo 'kaggle.json' descargado desde Kaggle.")
uploaded = files.upload()

if "kaggle.json" in uploaded:
    # Crear directorio de configuración de Kaggle
    kaggle_dir = os.path.expanduser("~/.kaggle")
    os.makedirs(kaggle_dir, exist_ok=True)

    # Definir rutas
    source_path = "kaggle.json"
    target_path = os.path.join(kaggle_dir, "kaggle.json")

    # Mover archivo a la ubicación correcta
    shutil.move(source_path, target_path)

    # Asignar permisos seguros
    os.chmod(target_path, 0o600)

    print("\nArchivo 'kaggle.json' configurado correctamente.")
    print("IMPORTANTE: Reinicia la sesión de Colab antes de continuar.")
else:
    print("No se subió el archivo 'kaggle.json'. Inténtalo de nuevo.")

Sube tu archivo 'kaggle.json' descargado desde Kaggle.


Saving kaggle.json to kaggle.json

Archivo 'kaggle.json' configurado correctamente.
IMPORTANTE: Reinicia la sesión de Colab antes de continuar.


# Explicación Ténica del paso:

- El archivo `kaggle.json` contiene las credenciales de acceso a la API de Kaggle.

- Se almacena en la ruta `~/.kaggle/kaggle.json`, que es la ubicación esperada por la librería oficial.

- Los permisos `0o600` garantizan que solo el usuario actual tenga acceso al archivo, cumpliendo con los requisitos de seguridad de Kaggle.

- Una vez completado este paso, es necesario reiniciar el entorno para habilitar correctamente la autenticación.

# 2. VERIFICACIÓN DE LA CONFIGURACIÓN DE KAGGLE

Se valida que el archivo `kaggle.json` se encuentre en la ruta correcta y que tenga los permisos adecuados.

In [5]:
import os

kaggle_json_path = os.path.expanduser("~/.kaggle/kaggle.json")

if os.path.exists(kaggle_json_path):
    print(f"El archivo existe en: {kaggle_json_path}")

    permissions = oct(os.stat(kaggle_json_path).st_mode & 0o777)
    print(f"Permisos actuales: {permissions}")

    if permissions == "0o600":
        print("La configuración de Kaggle es correcta.")
    else:
        print("Advertencia: los permisos no son los esperados (0o600).")
else:
    print("No se encontró el archivo 'kaggle.json'.")

El archivo existe en: /root/.kaggle/kaggle.json
Permisos actuales: 0o600
La configuración de Kaggle es correcta.


# 3. PREPARACIÓN DEL ENTORNO DE TRABAJO

Se crea una carpeta local para almacenar archivos descargados y resultados intermedios del análisis.



In [6]:
import os

os.makedirs("data", exist_ok=True)
print("Carpeta 'data' lista para trabajar.")

Carpeta 'data' lista para trabajar.


# 4. LIBRERÍAS PARA CARGA Y MANEJO DE DATOS

En esta fase únicamente se cargan librerías relacionadas con la obtención y manipulación inicial de datos.

Las librerías específicas de modelado se incorporarán en secciones posteriores del notebook.




In [7]:
!pip install -q kagglehub[pandas-datasets]

import pandas as pd
import numpy as np
import warnings
import matplotlib.pyplot as plt
import matplotlib

import kagglehub
from kagglehub import KaggleDatasetAdapter

warnings.filterwarnings("ignore")
matplotlib.rcParams["figure.dpi"] = 100

print("Librerías cargadas correctamente.")

Librerías cargadas correctamente.


# 5. CARGA DEL DATASET

Se carga el dataset `customer_shopping_data.csv` desde Kaggle.

Este dataset contiene información transaccional de ventas, incluyendo variables como fecha, categoría de producto, cantidad y precio.



In [8]:
file_path = "customer_shopping_data.csv"

df_empresa = kagglehub.load_dataset(
    KaggleDatasetAdapter.PANDAS,
    "mehmettahiraslan/customer-shopping-dataset",
    file_path
)

print("Dataset cargado correctamente.")
print("\nPrimeras 5 filas:")
display(df_empresa.head())

Using Colab cache for faster access to the 'customer-shopping-dataset' dataset.
Dataset cargado correctamente.

Primeras 5 filas:


,invoice_no,customer_id,gender,age,category,quantity,price,payment_method,invoice_date,shopping_mall
0,I138884,C241288,Female,28,Clothing,5,1500.40,Credit Card,5/8/2022,Kanyon
1,I317333,C111565,Male,21,Shoes,3,1800.51,Debit Card,12/12/2021,Forum Istanbul
2,I127801,C266599,Male,20,Clothing,1,300.08,Cash,9/11/2021,Metrocity
3,I173702,C988172,Female,66,Shoes,5,3000.85,Credit Card,16/05/2021,Metropol AVM
4,I337046,C189076,Female,53,Books,4,60.60,Cash,24/10/2021,Kanyon



# 6. INSPECCIÓN INICIAL DEL DATASET

Se realiza una revisión preliminar del dataset para analizar:
- Dimensiones
- Tipos de variables
- Presencia de valores nulos



In [9]:
print("Dimensiones del dataset:", df_empresa.shape)

print("\nTipos de datos:")
print(df_empresa.dtypes)

print("\nValores nulos por columna:")
print(df_empresa.isnull().sum())

Dimensiones del dataset: (99457, 10)

Tipos de datos:
invoice_no         object
customer_id        object
gender             object
age                 int64
category           object
quantity            int64
price             float64
payment_method     object
invoice_date       object
shopping_mall      object
dtype: object

Valores nulos por columna:
invoice_no        0
customer_id       0
gender            0
age               0
category          0
quantity          0
price             0
payment_method    0
invoice_date      0
shopping_mall     0
dtype: int64


In [11]:
# Resumen estadístico general
df_empresa.describe(include="all")

,invoice_no,customer_id,gender,age,category,quantity,price,payment_method,invoice_date,shopping_mall
count,99457,99457,99457,99457.000000,99457,99457.000000,99457.000000,99457,99457,99457
unique,99457,99457,2,NaN,8,NaN,NaN,3,797,10
top,I232867,C273973,Female,NaN,Clothing,NaN,NaN,Cash,24/11/2021,Mall of Istanbul
freq,1,1,59482,NaN,34487,NaN,NaN,44447,159,19943
mean,NaN,NaN,NaN,43.427089,NaN,3.003429,689.256321,NaN,NaN,NaN
std,NaN,NaN,NaN,14.990054,NaN,1.413025,941.184567,NaN,NaN,NaN
min,NaN,NaN,NaN,18.000000,NaN,1.000000,5.230000,NaN,NaN,NaN
25%,NaN,NaN,NaN,30.000000,NaN,2.000000,45.450000,NaN,NaN,NaN
50%,NaN,NaN,NaN,43.000000,NaN,3.000000,203.300000,NaN,NaN,NaN
75%,NaN,NaN,NaN,56.000000,NaN,4.000000,1200.320000,NaN,NaN,NaN


# 1. Análisis Exploratorio de Datos (EDA)

In [ ]:
tabla_desc = df_ventas_mensuales.groupby('categoria')['unidades_vendidas'].describe()
tabla_desc

NameError: name 'df_ventas_mensuales' is not defined

In [ ]:
# Top 10 categorías por volumen total de ventas
top10 = (
    df_ventas_mensuales
    .groupby('categoria')['unidades_vendidas']
    .sum()
    .sort_values(ascending=False)
    .head(10)
)

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

top10.plot(kind='bar', ax=axes[0])
axes[0].set_title('Top 10 Categorías por Unidades Vendidas (total histórico)')
axes[0].set_xlabel('Categoría')
axes[0].set_ylabel('Unidades vendidas')
axes[0].tick_params(axis='x', rotation=45)

evolucion = df_ventas_mensuales.groupby('fecha_venta')['unidades_vendidas'].sum()
evolucion.plot(ax=axes[1], marker='o')
axes[1].set_title('Evolución Mensual de Ventas Totales')
axes[1].set_xlabel('Mes')
axes[1].set_ylabel('Unidades vendidas')
axes[1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()



In [ ]:
# Histogramas de distribución

import matplotlib.pyplot as plt
import numpy as np

# Crear variable log (si no existe)
df_ventas_mensuales['cantidad_log'] = np.log1p(df_ventas_mensuales['unidades_vendidas'])

# Histograma original
counts, bins, patches = plt.hist(df_ventas_mensuales['unidades_vendidas'])

colores = ['pink', 'teal'] * len(patches)

for patch, color in zip(patches, colores):
    patch.set_facecolor(color)

plt.title("Distribución de unidades vendidas")
plt.xlabel("Unidades vendidas")
plt.ylabel("Frecuencia")
plt.show()

# Histograma logarítmico
counts, bins, patches = plt.hist(df_ventas_mensuales['cantidad_log'])

colores = ['green', 'purple'] * len(patches)

for patch, color in zip(patches, colores):
    patch.set_facecolor(color)

plt.title("Distribución logarítmica de unidades vendidas")
plt.xlabel("Log(unidades vendidas)")
plt.ylabel("Frecuencia")
plt.show()

In [ ]:
# Categorías únicas
num_categorias = df_ventas_mensuales['categoria'].nunique()
print(f'Número de categorías: {num_categorias}')

# Total de ventas por categoría (suma mensual acumulada)
ventas_por_categoria = (
    df_ventas_mensuales
    .groupby('categoria')['unidades_vendidas']
    .sum()
    .sort_values(ascending=False)
)

# Porcentaje acumulado (regla de Pareto)
porcentaje_acumulado = ventas_por_categoria.cumsum() / ventas_por_categoria.sum()
categorias_80 = porcentaje_acumulado[porcentaje_acumulado <= 0.8].index
print(f'Categorías que representan el 80% de las ventas: {len(categorias_80)}')

df_80 = df_ventas_mensuales[df_ventas_mensuales['categoria'].isin(categorias_80)]

# Boxplot mensual
plt.figure(figsize=(14, 6))
df_80.boxplot(column='unidades_vendidas', by='categoria', rot=90)
plt.title('Distribución Mensual de Unidades Vendidas (80% de las ventas)')
plt.suptitle('')
plt.xlabel('Categoría')
plt.ylabel('Unidades vendidas / mes')
plt.tight_layout()
plt.show()

print('\nEstadísticas descriptivas mensuales (80% de las ventas):')
stats_80 = (
    df_80
    .groupby('categoria')['unidades_vendidas']
    .describe()
    .round(2)
)
print(stats_80)

In [ ]:
# Distribución de unidades vendidas por día y categoría
# (se usa granularidad diaria para visualizar la cola de la distribución)
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

df_ventas_diarias['unidades_vendidas'].plot(
    kind='hist', bins=50, ax=axes[0], edgecolor='white')
axes[0].set_title('Distribución de Unidades Vendidas por Día y Categoría')
axes[0].set_xlabel('Unidades vendidas')
axes[0].set_ylabel('Frecuencia')

np.log1p(df_ventas_diarias['unidades_vendidas']).plot(
    kind='hist', bins=50, ax=axes[1], edgecolor='white')
axes[1].set_title('Distribución de Unidades Vendidas — Log-transform')
axes[1].set_xlabel('log(1 + unidades)')
axes[1].set_ylabel('Frecuencia')

plt.tight_layout()
plt.show()

El análisis exploratorio muestra que las ventas se concentran en un conjunto reducido de categorías, evidenciando una distribución desigual típica en entornos de comercio electrónico. Asimismo, la evolución temporal refleja variaciones a lo largo del tiempo, lo que justifica el uso de modelos de series temporales para capturar patrones y tendencias.

La distribución de las ventas presenta asimetría positiva, con presencia de valores extremos, lo cual se atenúa mediante la transformación logarítmica, facilitando su posterior modelado.

# 2. Preprocesamiento de Datos

Preparamos las variables para el entrenamiento de los modelos supervisados aplicando:

- **Log-transform**: `log1p` sobre `cantidad` para reducir el sesgo positivo observado en el EDA
- **Escalado**: `StandardScaler` sobre la variable temporal continua `tiempo`
- **Codificación de categóricas**: OneHotEncoding sobre `tipo_producto`.


Estas transformaciones mejoran la estabilidad numérica y la calidad de los modelos lineales.

In [ ]:
df_model = df_ventas_mensuales.copy()

df_model = df_model.rename(columns={
    'fecha_venta':     'fecha_hora',
    'categoria':       'tipo_producto',
    'unidades_vendidas': 'cantidad'
})

# fecha_hora ya es datetime (primer día de cada mes)
df_model['fecha_hora'] = pd.to_datetime(df_model['fecha_hora'])
df_model['año']  = df_model['fecha_hora'].dt.year
df_model['mes']  = df_model['fecha_hora'].dt.month

# Variable temporal continua: cada unidad = 1 mes
df_model['tiempo'] = df_model['año'] * 12 + df_model['mes']

df_model.head()

In [ ]:
# Agrupar por tiempo y tipo_producto

df_agg = (
    df_model
    .groupby(["tiempo", "tipo_producto"], as_index=False)
    .agg({"cantidad": "sum"})
)

df_agg.head()

##2.1 Log-Transform y Escalado


La distribución de cantidad presenta fuerte asimetría a la derecha (cola larga). Aplicamos **log1p** para acercarla a una distribución más estable, lo cual beneficia a los modelos lineales. Adicionalmente, escalamos **tiempo** con **StandardScaler** para evitar que su magnitud domine sobre las variables generadas posteriormente mediante codificación categórica.




In [ ]:
# Log-transform sobre cantidad
df_agg['cantidad_log'] = np.log1p(df_agg['cantidad'])

# Escalado de la variable temporal
scaler_tiempo = StandardScaler()
df_agg['tiempo_scaled'] = scaler_tiempo.fit_transform(df_agg[['tiempo']])

df_agg[['tiempo', 'tiempo_scaled', 'tipo_producto', 'cantidad', 'cantidad_log']].head(10)

##2.2 Codificación de variables categóricas (OneHotEncoding)

In [ ]:
# Encoder de categorías (útil para la regresión lineal y Random Forest, ARIMA no lo necesita)
encoder = OneHotEncoder(sparse_output=False, handle_unknown='ignore')
encoder.fit(df_agg[['tipo_producto']])

# Vista de las categorías codificadas
print(f'Categorías únicas: {len(encoder.categories_[0])}')
print('Primeras 5 categorías:', list(encoder.categories_[0][:5]))

En esta fase se prepararon las variables para el entrenamiento de los modelos supervisados. Para ello, se construyó una variable temporal continua (tiempo) a partir del año y el mes de la fecha de venta, permitiendo representar la evolución cronológica de las ventas.

Asimismo, se aplicó una transformación logarítmica sobre la variable **cantidad** con el objetivo de reducir la asimetría positiva observada en el análisis exploratorio. Finalmente, la variable temporal fue escalada mediante **StandardScaler**, mejorando la estabilidad numérica y favoreciendo el desempeño de modelos sensibles a la escala, como la regresión lineal.

# 3. Modelo de Regresión Lineal

**Justificación del algoritmo:**
La Regresión Lineal es un modelo base interpretable que permite cuantificar la relación entre el tiempo (variable continua) y la cantidad vendida por categoría de producto. Al incorporar el tipo de producto mediante OneHotEncoding, el modelo aprende una tendencia lineal independiente por categoría.

Es un punto de partida sólido y su interpretabilidad facilita la validación de resultados con el negocio.

In [ ]:
# Variable independiente: tiempo escalado + categoría codificada
X = np.hstack([
    df_agg[['tiempo_scaled']].values,
    encoder.transform(df_agg[['tipo_producto']])
])

# Variable objetivo: cantidad en escala log
y = df_agg['cantidad_log'].values

print('Shape X:', X.shape)
print('Shape y:', y.shape)

## 3.1 División Train / Test (Temporal)

Dividimos el dataset respetando el orden temporal: el **80%** más antiguo para entrenamiento y el **20%** más reciente para test. Esto evita el data leakage que ocurriría con una división aleatoria en datos de series temporales.

In [ ]:
# División temporal: 80% train, 20% test
tiempos_unicos = sorted(df_agg['tiempo'].unique())
split_idx = int(len(tiempos_unicos) * 0.8)
tiempo_corte = tiempos_unicos[split_idx]

mask_train = df_agg['tiempo'] <= tiempo_corte
mask_test  = df_agg['tiempo']  > tiempo_corte

X_train, X_test = X[mask_train], X[mask_test]
y_train, y_test = y[mask_train], y[mask_test]

print(f'Tiempo de corte: {tiempo_corte}')
print(f'Registros en train: {X_train.shape[0]}')
print(f'Registros en test:  {X_test.shape[0]}')

## 3.2 Entrenamiento

In [ ]:
from sklearn.linear_model import LinearRegression
modelo = LinearRegression()
modelo.fit(X_train, y_train)

## 3.3 Evaluación y Métricas

Evaluamos el modelo sobre el conjunto de test con tres métricas:
- **MAE** (Mean Absolute Error): error medio absoluto
- **RMSE** (Root Mean Squared Error): penaliza errores grandes
- **R²**: proporción de varianza explicada (1.0 = perfecto, 0 = no mejor que la media)

In [ ]:
# Predicciones sobre test
y_pred_lr = modelo.predict(X_test)

# Métricas en escala log
mae_lr   = mean_absolute_error(y_test, y_pred_lr)
rmse_lr  = np.sqrt(mean_squared_error(y_test, y_pred_lr))
r2_lr    = r2_score(y_test, y_pred_lr)

print('=== Métricas Regresión Lineal (escala log) ===')
print(f'MAE  : {mae_lr:.4f}')
print(f'RMSE : {rmse_lr:.4f}')
print(f'R2   : {r2_lr:.4f}')

# Métricas en escala original (invertir log-transform)
y_pred_orig = np.expm1(y_pred_lr)
y_test_orig = np.expm1(y_test)
mae_orig  = mean_absolute_error(y_test_orig, y_pred_orig)
rmse_orig = np.sqrt(mean_squared_error(y_test_orig, y_pred_orig))
print('\n=== Métricas Regresión Lineal (escala original) ===')
print(f'MAE  : {mae_orig:.2f} unidades')
print(f'RMSE : {rmse_orig:.2f} unidades')


## 3.4 Predicción Futura (próximos 3 meses)

In [ ]:
from sklearn.linear_model import LinearRegression

# Lista para almacenar las predicciones de cada producto
all_future_predictions = []
productos = df_agg['tipo_producto'].unique()
forecast_steps = 6

# Iterar sobre cada producto para entrenar un modelo y predecir individualmente
for producto in productos:
    df_p = df_agg[df_agg['tipo_producto'] == producto].copy()

    if len(df_p) < 2: # Se necesitan al menos 2 puntos para una regresión lineal simple
        continue

    X = df_p[['tiempo_scaled']].values
    y = df_p['cantidad_log'].values

    # Entrenar un nuevo modelo de Regresión Lineal para este producto
    modelo_prod = LinearRegression()
    modelo_prod.fit(X, y)

    ultimo_tiempo_prod = df_p['tiempo'].max()

    futuros_tiempos_raw = np.array([ultimo_tiempo_prod + i for i in range(1, forecast_steps + 1)])

    futuros_tiempos_scaled = scaler_tiempo.transform(futuros_tiempos_raw.reshape(-1, 1))

    predicciones_log = modelo_prod.predict(futuros_tiempos_scaled)
    predicciones_orig = np.expm1(predicciones_log).clip(0)


    df_pred_prod = pd.DataFrame({
        'tiempo': futuros_tiempos_raw,
        'tiempo_scaled': futuros_tiempos_scaled.flatten(),
        'tipo_producto': producto,
        'cantidad_predicha': predicciones_orig
    })
    all_future_predictions.append(df_pred_prod)


df_futuro = pd.concat(all_future_predictions, ignore_index=True)

df_futuro.head(10)

## 3.5 Visualización (Top 10 Categorías por Volumen)

In [ ]:
def tiempo_a_fecha(t):
    """Convierte tiempo=año*12+mes a string 'YYYY-MM'."""
    anio = t // 12
    mes  = t % 12
    if mes == 0:
        mes = 12
        anio -= 1
    return pd.Timestamp(year=anio, month=mes, day=1)

top10_cats = (
    df_agg.groupby('tipo_producto')['cantidad'].sum()
    .sort_values(ascending=False)
    .head(10)
    .index
)

fig, axes = plt.subplots(5, 2, figsize=(16, 25))
axes = axes.flatten()

for idx, producto in enumerate(top10_cats):
    subset_real = df_agg[df_agg['tipo_producto'] == producto].copy()
    subset_pred = df_futuro[df_futuro['tipo_producto'] == producto].copy()

    # Convertir 'tiempo' a formato de fecha para el eje X
    subset_real['fecha_mes'] = subset_real['tiempo'].apply(tiempo_a_fecha).dt.strftime('%Y-%m')
    subset_pred['fecha_mes'] = subset_pred['tiempo'].apply(tiempo_a_fecha).dt.strftime('%Y-%m')

    # Asegurar el orden cronológico para el gráfico
    subset_real = subset_real.sort_values(by='fecha_mes')
    subset_pred = subset_pred.sort_values(by='fecha_mes')

    ax = axes[idx]
    ax.plot(subset_real['fecha_mes'], np.expm1(subset_real['cantidad_log']), # Usar fecha_mes
            label='Real', marker='o', markersize=3)
    ax.plot(subset_pred['fecha_mes'], subset_pred['cantidad_predicha'], # Usar fecha_mes
            label='Prediccion', linestyle='--', marker='s', markersize=4, color='tomato')
    ax.set_title(producto)
    ax.set_xlabel('Fecha (Año-Mes)') # Cambiar etiqueta del eje X
    ax.set_ylabel('Cantidad')
    ax.legend(fontsize=8)
    ax.tick_params(axis='x', rotation=45) # Rotar y alinear etiquetas

plt.suptitle('Regresion Lineal: Ventas Reales vs Prediccion Futura (Top 10)', fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

## 3.6 Interpretación

La regresión lineal captura tendencias generales de crecimiento o descenso por categoría. Las categorías con mayor R² presentan una tendencia temporal clara y estable, mientras que aquellas con R² bajo exhiben mayor volatilidad o comportamiento no lineal.

**Limitación principal:** El modelo asume una relación lineal con el tiempo y no captura estacionalidad ni patrones complejos.

Los resultados gráficos muestran que la regresión lineal captura adecuadamente la tendencia general de crecimiento en las distintas categorías de producto. Sin embargo, se observa que el modelo tiende a sobreestimar las ventas futuras, especialmente en categorías con alta variabilidad.

Esto se debe a que la regresión lineal asume una relación estrictamente lineal con el tiempo, lo que impide capturar fluctuaciones, estacionalidad o comportamientos no lineales presentes en los datos reales.

En consecuencia, aunque el modelo resulta útil como línea base por su interpretabilidad, presenta limitaciones claras para modelar dinámicas complejas, lo cual justifica el uso de modelos más avanzados en etapas posteriores del proyecto.

# 4. Modelo ARIMA (AutoRegressive Integrated Moving Average) **(Cambiar por Sarima)**

**Justificación del algoritmo:**
ARIMA es un modelo estadístico diseñado para series temporales univariadas. A diferencia de la regresión lineal, modela explícitamente:

- **AR(p)**: autocorrelación — las ventas dependen de sus valores pasados
- **I(d)**: diferenciación para hacer la serie estacionaria
- **MA(q)**: componente de media móvil del error de predicción

Se aplica individualmente por categoría de producto, lo que permite capturar patrones específicos de cada tipo. La configuración **(p=1, d=1, q=1)** es una elección estándar y robusta para series de demanda comercial, especialmente como aproximación inicial en un contexto de MVP.

In [ ]:

# Se parte de df_model que ya tiene granularidad mensual
df_arima = df_model[['fecha_hora', 'tipo_producto', 'cantidad']].copy()
df_arima = df_arima.sort_values('fecha_hora').reset_index(drop=True)

In [ ]:
df_arima.head()

In [ ]:
# Obtener categorías de los productos
categorias = df_arima['tipo_producto'].unique()

In [ ]:
df_arima = df_arima[['fecha_hora','tipo_producto','cantidad']]

In [ ]:
# Instalar pmdarima para selección automática de orden ARIMA
!pip install pmdarima -q
from pmdarima import auto_arima

In [ ]:
resultados = {}

CONFIG = {
    'forecast_steps': 3,    # pronosticar 3 meses hacia adelante
    'min_datos': 12,        # mínimo 12 meses para modelar
    'test_size':  4,        # últimos 4 meses como test (~20% de ~24 meses)
    'freq': 'MS'            # frecuencia mensual (Month Start)
}

categorias = df_arima['tipo_producto'].unique()

for categoria in categorias:

    df_prod = df_arima[df_arima['tipo_producto'] == categoria]
    serie = df_prod.groupby('fecha_hora')['cantidad'].sum().sort_index()
    serie = serie.asfreq(CONFIG['freq']).fillna(0)

    if len(serie) < CONFIG['min_datos']:
        continue
    if len(serie) <= CONFIG['test_size']:
        continue

    train = serie.iloc[:-CONFIG['test_size']]
    test  = serie.iloc[-CONFIG['test_size']:]

    try:
        modelo_auto = auto_arima(
            train,
            start_p=0, max_p=3,
            start_q=0, max_q=3,
            d=None,
            seasonal=False,
            information_criterion='aic',
            stepwise=True,
            suppress_warnings=True,
            error_action='ignore'
        )
        order_seleccionado = modelo_auto.order

        from statsmodels.tsa.arima.model import ARIMA as ARIMA_sm
        modelo_sm = ARIMA_sm(train, order=order_seleccionado).fit()

        pred            = modelo_sm.forecast(steps=len(test))
        mae             = np.mean(np.abs(test.values - pred.values))
        rmse            = np.sqrt(np.mean((test.values - pred.values)**2))
        forecast_result = modelo_sm.get_forecast(steps=CONFIG['forecast_steps'])
        forecast_mean   = forecast_result.predicted_mean
        forecast_ci     = forecast_result.conf_int(alpha=0.20)

        resultados[categoria] = {
            'serie':       serie,
            'train':       train,
            'test':        test,
            'pred':        pred,
            'forecast':    forecast_mean,
            'forecast_ci': forecast_ci,
            'order':       order_seleccionado,
            'MAE':         mae,
            'RMSE':        rmse
        }
        print(f'{categoria:35s}  order={order_seleccionado}  MAE={mae:.1f}  RMSE={rmse:.1f}')

    except Exception as e:
        print(f'Error en {categoria}: {e}')


## 4.1 Visualización por Categoría (Top 10 por Volumen)

In [ ]:
# Ordenar resultados por volumen total y mostrar top 10
resultados_sorted = sorted(
    resultados.items(),
    key=lambda x: x[1]['serie'].sum(),
    reverse=True
)[:10]

for categoria, res in resultados_sorted:

    fig, ax = plt.subplots(figsize=(13, 5))

    ax.plot(res['train'], label='Train', color='steelblue', lw=1.5)
    ax.plot(res['test'],  label='Test (real)', color='seagreen', lw=2)
    ax.plot(res['pred'],  linestyle='--', label=f'Predicción test', color='tomato', lw=1.8)
    ax.plot(res['forecast'], linestyle=':', label='Forecast futuro', color='orange', lw=2)

    # Intervalo de confianza 80% para el forecast
    ci = res['forecast_ci']
    ax.fill_between(
        ci.index,
        ci.iloc[:, 0].clip(0),
        ci.iloc[:, 1],
        alpha=0.25, color='orange', label='IC 80%'
    )

    ax.set_title(
        f'ARIMA{res["order"]} — {categoria}  |  MAE={res["MAE"]:.2f}  RMSE={res["RMSE"]:.2f}',
        fontsize=11
    )
    ax.set_xlabel('Fecha')
    ax.set_ylabel('Cantidad')
    ax.legend(fontsize=9)
    plt.tight_layout()
    plt.show()


## 4.2 Tabla Resumen de Métricas ARIMA

In [ ]:
resumen_arima = pd.DataFrame([
    {
        'Categoria': cat,
        'N_obs': len(res['serie']),
        'MAE': round(res['MAE'], 3),
        'RMSE': round(res['RMSE'], 3)
    }
    for cat, res in resultados.items()
]).sort_values('MAE').reset_index(drop=True)

print(f'Categorias modeladas con ARIMA: {len(resumen_arima)}')
print(f'MAE promedio : {resumen_arima["MAE"].mean():.3f}')
print(f'RMSE promedio: {resumen_arima["RMSE"].mean():.3f}')

resumen_arima.head(20)

##4.3 Interpretación

Los modelos **ARIMA (1,1,1)** capturan la dinámica de corto plazo de cada categoría, aprovechando la dependencia temporal entre observaciones consecutivas. Las categorías con **MAE bajo** presentan series más estables y predecibles, mientras que aquellas con alta variabilidad, cambios bruscos o bajo volumen de ventas tienden a obtener peores métricas.

En comparación con la regresión lineal, **ARIMA explota de forma más directa la estructura temporal de la serie**, por lo que resulta más adecuado para predicciones de corto plazo en contextos donde la dependencia con valores pasados es relevante. En cambio, la regresión lineal funciona mejor como modelo base interpretable para capturar tendencias generales de largo plazo.

En el flujo metodológico del proyecto, primero se utiliza la **regresión lineal** como línea base por su simplicidad e interpretabilidad. Posteriormente, se introduce **ARIMA** como modelo temporal clásico, capaz de capturar dependencias dinámicas más complejas. Finalmente, se incorpora **Random Forest** con el objetivo de evaluar la capacidad de técnicas de aprendizaje automático para modelar relaciones no lineales y mejorar la precisión predictiva.

##4.4 Comparación entre Regresión Lineal y ARIMA

In [ ]:
# La comparación completa de los tres modelos (Ridge, ARIMA, Random Forest)
# se presenta en la Sección 5.3 una vez entrenado el Random Forest.
print(f'Métricas Ridge  — MAE: {mae_orig:.2f} | RMSE: {rmse_orig:.2f} | R²: {r2_lr:.4f}')
print(f'Métricas ARIMA  — MAE: {resumen_arima["MAE"].mean():.2f} | RMSE: {resumen_arima["RMSE"].mean():.2f}')
print('→ Comparación final de los 3 modelos disponible en Sección 5.3')

# 5. Modelo Avanzado: Random Forest Regressor (Ensemble) **(Cambiar por redes neuronales o XGBoost)**

**Justificación:**
Random Forest es un modelo ensemble que combina múltiples árboles de decisión entrenados con subconjuntos aleatorios de datos y features (bagging). Sus ventajas frente a la regresión lineal son:

- Captura relaciones **no lineales** entre tiempo y ventas
- Es **robusto a outliers** y no requiere distribución normal de los residuos
- Proporciona **importancia de variables**, útil para interpretación de negocio
- Permite **ajuste de hiperparámetros** con `GridSearchCV` y validación cruzada temporal

Este modelo representa la extensión avanzada requerida y permite contrastar un enfoque clásico de series temporales con una técnica de aprendizaje automático más flexible

## 5.1 Preparación de Datos

In [ ]:

df_rf = df_agg.copy()

# Features temporales adicionales
df_rf['anio']    = (df_rf['tiempo'] - 1) // 12
df_rf['mes_num'] = ((df_rf['tiempo'] - 1) % 12) + 1

# Reutilizar el encoder global fiteado en la Sección 2
cat_encoded = encoder.transform(df_rf[['tipo_producto']])
df_cat = pd.DataFrame(
    cat_encoded,
    columns=encoder.get_feature_names_out(['tipo_producto']),
    index=df_rf.index
)

X_rf = pd.concat([df_rf[['tiempo_scaled', 'anio', 'mes_num']], df_cat], axis=1)
y_rf = df_rf['cantidad_log'].values

print('Shape X_rf:', X_rf.shape)

## 5.2 Ajuste de Hiperparámetros con GridSearchCV

In [ ]:
# División temporal 80/20 — mismo corte que Regresión Lineal
X_rf_train = X_rf[mask_train]
X_rf_test  = X_rf[mask_test]
y_rf_train = y_rf[mask_train]
y_rf_test  = y_rf[mask_test]

print(f'Tiempo de cort: {tiempo_corte}')
print(f'Train RF: {X_rf_train.shape[0]} registros | Test RF: {X_rf_test.shape[0]} registros')

# GridSearchCV con validación cruzada temporal
param_grid = {
    'n_estimators':     [100, 200],
    'max_depth':        [5, 10, None],
    'min_samples_leaf': [2, 5]
}

tscv = TimeSeriesSplit(n_splits=3)
rf_base = RandomForestRegressor(random_state=42, n_jobs=-1)

grid_search = GridSearchCV(
    rf_base,
    param_grid,
    cv=tscv,
    scoring='neg_mean_absolute_error',
    n_jobs=-1,
    verbose=0
)

grid_search.fit(X_rf_train, y_rf_train)

print('Mejores hiperparámetros:', grid_search.best_params_)
print(f'Mejor MAE (CV): {-grid_search.best_score_:.4f}')

## 5.3 Evaluación y Comparación con Regresión Lineal

In [ ]:
modelo_rf = grid_search.best_estimator_

y_rf_pred = modelo_rf.predict(X_rf_test)

mae_rf  = mean_absolute_error(y_rf_test, y_rf_pred)
rmse_rf = np.sqrt(mean_squared_error(y_rf_test, y_rf_pred))
r2_rf   = r2_score(y_rf_test, y_rf_pred)

print('=== Comparacion de Modelos (escala log) ===')
comparacion = pd.DataFrame({
    'Modelo': ['Regresion Lineal', 'ARIMA', 'Random Forest'],
    'MAE':  [round(mae_lr, 4), round(resumen_arima['MAE'].mean(), 4), round(mae_rf, 4)],
    'RMSE': [round(rmse_lr, 4), round(resumen_arima['RMSE'].mean(), 4), round(rmse_rf, 4)],
    'R2':   [round(r2_lr, 4), np.nan, round(r2_rf, 4)]
})
print(comparacion.to_string(index=False))

## 5.4 Visualizaciones: Predicho vs Real — Random Forest

---

Se presentan tres tipos de visualización para evaluar la calidad del modelo:
1. **Scatter plot** predicho vs rea
2. **Serie temporal** por categoría — real vs predicho en el conjunto de test
3. **Distribución de residuales** (error = real − predicho)

In [ ]:
# ── 1. Scatter plot global: Predicho vs Real (escala original) ──────────
y_rf_orig      = np.expm1(y_rf_test)
y_rf_pred_orig = np.expm1(y_rf_pred).clip(0)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Scatter
ax = axes[0]
ax.scatter(y_rf_orig, y_rf_pred_orig, alpha=0.4, s=20, color='steelblue')
lim = max(y_rf_orig.max(), y_rf_pred_orig.max()) * 1.05
ax.plot([0, lim], [0, lim], 'r--', lw=1.5, label='Predicción perfecta')
ax.set_xlabel('Real (unidades)')
ax.set_ylabel('Predicho (unidades)')
ax.set_title('Random Forest — Predicho vs Real (escala original)')
ax.legend()

# Residuales
residuales = y_rf_orig - y_rf_pred_orig
ax2 = axes[1]
ax2.hist(residuales, bins=50, color='steelblue', edgecolor='white')
ax2.axvline(0, color='red', linestyle='--', lw=1.5)
ax2.set_xlabel('Error (Real − Predicho)')
ax2.set_ylabel('Frecuencia')
ax2.set_title('Distribución de Residuales — Random Forest')

plt.tight_layout()
plt.show()

print(f'Sesgo medio (bias): {residuales.mean():.4f}')
print(f'Std residuales    : {residuales.std():.4f}')

In [ ]:
def tiempo_a_fecha(t):
    """Convierte tiempo=año*12+mes a string 'YYYY-MM'."""
    anio = t // 12
    mes  = t % 12
    if mes == 0:
        mes = 12
        anio -= 1
    return pd.Timestamp(year=anio, month=mes, day=1)

In [ ]:
# ── 2. Serie temporal mensual: Real vs Predicho por categoría (Top 10) ──
df_rf_test_eval = df_rf[mask_test].copy()
df_rf_test_eval['y_real']    = np.expm1(y_rf_test)
df_rf_test_eval['y_pred']    = np.expm1(y_rf_pred).clip(0)
df_rf_test_eval['error_abs'] = np.abs(df_rf_test_eval['y_real'] - df_rf_test_eval['y_pred'])
df_rf_test_eval['fecha']     = df_rf_test_eval['tiempo'].apply(tiempo_a_fecha)

top10_rf = (
    df_rf.groupby('tipo_producto')['cantidad']
    .sum()
    .sort_values(ascending=False)
    .head(10)
    .index
)

fig, axes = plt.subplots(5, 2, figsize=(16, 26))
axes = axes.flatten()

for idx, producto in enumerate(top10_rf):
    sub = df_rf_test_eval[df_rf_test_eval['tipo_producto'] == producto].sort_values('fecha')
    ax  = axes[idx]
    ax.plot(sub['fecha'], sub['y_real'], marker='o', ms=4,
            label='Real', color='seagreen', lw=1.5)
    ax.plot(sub['fecha'], sub['y_pred'], marker='s', ms=4,
            linestyle='--', label='Predicho (RF)', color='tomato', lw=1.5)
    mae_cat = sub['error_abs'].mean()
    ax.set_title(f'{producto}  (MAE={mae_cat:.1f})', fontsize=9)
    ax.set_xlabel('Mes')
    ax.set_ylabel('Unidades vendidas')
    ax.tick_params(axis='x', rotation=30)
    ax.legend(fontsize=8)

plt.suptitle('Random Forest — Real vs Predicho por Mes (Top 10 categorías)',
             fontsize=13, y=1.01)
plt.tight_layout()
plt.show()


## 5.4 Importancia de Variables

In [ ]:
feature_names = list(X_rf.columns)
importancias = pd.Series(modelo_rf.feature_importances_, index=feature_names)
top_features = importancias.sort_values(ascending=False).head(15)

plt.figure(figsize=(10, 6))
top_features.plot(kind='barh', color='steelblue')
plt.title('Importancia de Variables — Random Forest (Top 15)')
plt.xlabel('Importancia')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

print('tiempo_scaled y mes_num capturan la dinamica temporal,')
print('las variables de categoria reflejan el impacto diferencial por tipo de producto.')

##5.5 Interpretación


Random Forest permite capturar relaciones no lineales entre el tiempo y la cantidad vendida, lo que le otorga una ventaja frente a la regresión lineal cuando las series presentan patrones más complejos o comportamientos no estrictamente crecientes. Además, al incorporar variables temporales y la categoría del producto, el modelo puede aprender diferencias específicas entre segmentos sin imponer una forma funcional fija.

En comparación con la regresión lineal, Random Forest suele ofrecer mejor desempeño predictivo cuando existen interacciones, cambios abruptos o heterogeneidad entre categorías. Frente a ARIMA, presenta la ventaja de integrar múltiples variables explicativas y no depender exclusivamente de la estructura autorregresiva de una serie univariada.

La interpretación de importancia de variables permite identificar qué factores influyen más en la predicción, reforzando la utilidad del modelo no solo desde la precisión, sino también desde la comprensión del comportamiento de ventas por categoría.

#7. Uso de ChatGPT como herramienta de apoyo y comunicación

**Justificación:** Se incorpora ChatGPT como herramienta complementaria para la **interpretación y comunicación de resultados** obtenidos a partir de los modelos predictivos. Su función no consiste en generar predicciones ni sustituir los modelos estadísticos o de machine learning, sino en **traducir métricas técnicas y hallazgos cuantitativos en explicaciones comprensibles para usuarios no especializados.**

Esta integración aporta valor al sistema por tres razones principales:

*   Facilita la interpretación de métricas como **MAE, RMSE y R².**
*   Mejora la comunicación de resultados hacia usuarios con perfil no técnico.
*   Refuerza el enfoque práctico del TFM, orientado a pequeñas y medianas empresas.


##7.1 Configuración de la librería y autenticación

En esta versión se implementa una **simulación funcional del comportamiento esperado de ChatGPT**, evitando dependencia de cuota o facturación en el entorno de desarrollo.

In [ ]:
# En la versión simulada no es obligatorio usar la API de OpenAI
# Se mantiene una estructura compatible con una futura integración real

import os

##7.2 Función para interpretar resultados de modelos

In [ ]:
def interpretar_resultados_modelo(modelo, mae, rmse, r2=None, contexto="predicción de ventas por categoría"):
    texto = f"""
=== Interpretación simulada tipo ChatGPT ===

Modelo: {modelo}
Contexto: {contexto}

1. Interpretación técnica breve:
El modelo presenta un MAE de {mae:.2f} y un RMSE de {rmse:.2f}.
"""

    if r2 is not None:
        texto += f" El valor de R² es {r2:.2f}, lo que permite evaluar la proporción de varianza explicada por el modelo.\n"
    else:
        texto += " En este caso no se dispone de R², ya que el modelo evaluado no utiliza esta métrica como referencia principal.\n"

    texto += f"""
2. Explicación sencilla orientada a negocio:
El modelo {modelo} permite estimar el comportamiento futuro de las ventas con un nivel razonable de precisión, sirviendo como apoyo para planificación, control de stock y toma de decisiones.

3. Limitación principal del modelo:
Aunque ofrece información útil, este modelo puede no capturar completamente cambios bruscos, estacionalidad compleja o relaciones altamente no lineales presentes en los datos.
"""

    return texto

##7.3 Interpretación automática de la Regresión Lineal

In [ ]:
texto_lr = interpretar_resultados_modelo(
    modelo="Regresión Lineal",
    mae=mae_orig,
    rmse=rmse_orig,
    r2=r2_lr,
    contexto="forecasting de ventas de productos de e-commerce"
)

print("=== Interpretación tipo ChatGPT (simulada): Regresión Lineal ===")
print(texto_lr)

##7.4 Interpretación automática de ARIMA

In [ ]:
texto_arima = interpretar_resultados_modelo(
    modelo="ARIMA (1,1,1)",
    mae=resumen_arima["MAE"].mean(),
    rmse=resumen_arima["RMSE"].mean(),
    r2=None,
    contexto="forecasting temporal de ventas por categoría"
)

print("=== Interpretación ChatGPT: ARIMA ===")
print(texto_arima)

##7.5 Interpretación automática de Random Forest

In [ ]:
texto_rf = interpretar_resultados_modelo(
    modelo="Random Forest",
    mae=mae_rf,
    rmse=rmse_rf,
    r2=r2_rf,
    contexto="forecasting de ventas de productos de e-commerce"
)

print("=== Interpretación ChatGPT: Random Forest ===")
print(texto_rf)

##7.6 Almacenamiento de interpretaciones

In [ ]:
with open("interpretacion_modelos.txt", "w", encoding="utf-8") as f:
    f.write("=== Regresión Lineal ===\n")
    f.write(texto_lr + "\n\n")
    f.write("=== ARIMA ===\n")
    f.write(texto_arima + "\n\n")
    f.write("=== Random Forest ===\n")
    f.write(texto_rf + "\n")

##7.7 Integración en Streamlit para mostrar la interpretación

In [ ]:
# Instalar Streamlit si no está instalado
!pip install streamlit

import streamlit as st
import pandas as pd

# -------------------------------
# Función simulada tipo ChatGPT
# -------------------------------

def interpretar_streamlit(modelo, mae, rmse, r2=None):
    texto = f"""
### Interpretación tipo ChatGPT

**Modelo seleccionado:** {modelo}

#### 1. Interpretación técnica
El modelo presenta un **MAE de {mae:.2f}** y un **RMSE de {rmse:.2f}**.
"""

    if r2 is not None:
        texto += f" Además, el **R² es {r2:.2f}**, lo que permite evaluar la proporción de varianza explicada por el modelo.\n"
    else:
        texto += " En este caso no se dispone de **R²**, ya que este modelo no utiliza dicha métrica como referencia principal.\n"

    texto += f"""
#### 2. Explicación sencilla para negocio
El modelo **{modelo}** permite estimar el comportamiento futuro de las ventas con un nivel razonable de precisión, sirviendo como apoyo para planificación, control de inventario y toma de decisiones.

#### 3. Limitación principal
Aunque ofrece información útil, este modelo puede no capturar completamente cambios bruscos, estacionalidad compleja o relaciones altamente no lineales presentes en los datos.
"""

    return texto


# -------------------------------
# Configuración de página
# -------------------------------

st.set_page_config(page_title="Interpretación de Modelos", layout="wide")

st.title("Interpretación automática de resultados")
st.write(
    "Esta sección muestra una interpretación tipo ChatGPT para facilitar la comprensión "
    "de los resultados de los modelos predictivos."
)

# -------------------------------
# Diccionario de métricas
# Usa aquí las métricas reales ya calculadas
# -------------------------------

metricas_modelos = {
    "Regresión Lineal": {
        "MAE": mae_orig,
        "RMSE": rmse_orig,
        "R2": r2_lr
    },
    "ARIMA": {
        "MAE": resumen_arima["MAE"].mean(),
        "RMSE": resumen_arima["RMSE"].mean(),
        "R2": None
    },
    "Random Forest": {
        "MAE": mae_rf,
        "RMSE": rmse_rf,
        "R2": r2_rf
    }
}

# -------------------------------
# Selector de modelo
# -------------------------------

modelo_seleccionado = st.selectbox(
    "Selecciona el modelo a interpretar",
    list(metricas_modelos.keys())
)

mae_val = metricas_modelos[modelo_seleccionado]["MAE"]
rmse_val = metricas_modelos[modelo_seleccionado]["RMSE"]
r2_val = metricas_modelos[modelo_seleccionado]["R2"]

# -------------------------------
# Mostrar métricas en tarjetas
# -------------------------------

col1, col2, col3 = st.columns(3)

with col1:
    st.metric("MAE", f"{mae_val:.2f}")

with col2:
    st.metric("RMSE", f"{rmse_val:.2f}")

with col3:
    if r2_val is not None:
        st.metric("R²", f"{r2_val:.2f}")
    else:
        st.metric("R²", "N/A")

# -------------------------------
# Botón para generar interpretación
# -------------------------------

if st.button("Generar interpretación"):
    with st.spinner("Generando interpretación..."):
        texto = interpretar_streamlit(
            modelo=modelo_seleccionado,
            mae=mae_val,
            rmse=rmse_val,
            r2=r2_val
        )

    st.markdown("---")
    st.chat_message("assistant").markdown(texto)

# -------------------------------
# Tabla opcional de métricas
# -------------------------------

st.markdown("---")
st.subheader("Resumen comparativo de métricas")

tabla_metricas = pd.DataFrame(metricas_modelos).T.reset_index()
tabla_metricas.columns = ["Modelo", "MAE", "RMSE", "R2"]

st.dataframe(tabla_metricas, use_container_width=True)

En la aplicación desarrollada con Streamlit se incorporó un módulo de interpretación automática de resultados, orientado a traducir métricas técnicas en explicaciones comprensibles para el usuario final. Esta funcionalidad actúa como una capa complementaria de comunicación, mejorando la accesibilidad del sistema sin intervenir en el proceso de modelado predictivo.

##7.8 Interpretación metodológica

La incorporación de ChatGPT en el sistema se plantea como una **capa de apoyo interpretativo**, orientada a facilitar la lectura de resultados y la comunicación con el usuario final. En este sentido, no reemplaza el proceso de modelado, sino que complementa la aplicación al convertir métricas y predicciones en explicaciones comprensibles.

En esta implementación, la funcionalidad se presenta mediante una **simulación estructurada del comportamiento esperado**, con el fin de mantener la lógica del sistema sin depender de restricciones externas de cuota o facturación. Esto permite demostrar la viabilidad de la integración dentro del MVP y su valor práctico en contextos empresariales donde la interpretabilidad resulta tan importante como la precisión predictiva

La simulación implementada puede ser reemplazada posteriormente por una integración real con la API de OpenAI, manteniendo la misma estructura funcional del sistema.